In [ ]:
!pip install -q transformers datasets jiwer torch librosa soundfile accelerate qwen_asr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.6/141.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 90.3 MB/s eta 0:00:00


In [1]:
import logging
import warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

from torch.utils.data import Dataset
from datasets import load_dataset, Audio
from typing import Tuple, Union, Optional
import glob
import json
from pathlib import Path
from tqdm import tqdm
import torch
import torchaudio
import soundfile as sf
import io
import os
from torch.utils.data import Dataset
from datasets import load_dataset, Audio
from typing import Tuple, Union, Optional
import glob
import json
from pathlib import Path
from tqdm import tqdm

In [2]:
import unicodedata
VI_ALPHABET = (
    "aàảãáạăằẳẵắặâầẩẫấậbcdđeèẻẽéẹêềểễếệghiìỉĩíị"
    "klmnoòỏõóọôồổỗốộơờởỡớợpqrstuùủũúụưừửữứựvxyỳỷỹýỵ"
    " '"
)

def normalize_transcript(text: str) -> str:
    """Lowercase, NFC normalize, remove special chars; preserve Vietnamese diacritics."""
    text = unicodedata.normalize("NFC", text.lower().strip())
    return "".join(c for c in text if c in VI_ALPHABET)

In [3]:
class ViMD(Dataset):
    """
    ViMD Dataset class thiết kế để thay thế LIBRISPEECH.
    Trả về tuple: (waveform, sample_rate, transcript)
    """

    def __init__(
        self,
        dataset_name: str = "ViMD",  
        split: str = "train",       
        target_sample_rate: Optional[int] = 16000,
        min_duration_sec: float = 0.3,
        max_silence_ratio: float = 0.95,
        filter_bad_samples: bool = True,
    ) -> None:
        self.dataset_name = dataset_name
        self.split = split
        self.target_sr = target_sample_rate
        self.min_duration_sec = min_duration_sec
        self.max_silence_ratio = max_silence_ratio
        self.filter_bad_samples = filter_bad_samples

        # Load dataset từ Hugging Face (giữ nguyên cấu trúc decode=False để xử lý bytes)
        print(f"Loading ViMD dataset split: {split}...")
#-------------------------------------------------------------KHÚC NÀY ĐÃ SỬA---------------------------------------------------        
        # self._dataset = load_dataset(dataset_name, split=split)
        if split == 'train':
            data_dir = r"D:\hf_datasets\hub\datasets--nguyendv02--ViMD_Dataset\snapshots\3a5b30157034e7eadd5c75fae1a820c6f9383398\data"
            train_files = sorted(glob.glob(os.path.join(data_dir, "train-*.parquet")))

            self._dataset = load_dataset(
                "parquet",
                data_files=train_files,
                split="train"
            )
        if split == 'test':
            data_dir = r"D:\hf_datasets\hub\datasets--nguyendv02--ViMD_Dataset\snapshots\3a5b30157034e7eadd5c75fae1a820c6f9383398\data"
            train_files = sorted(glob.glob(os.path.join(data_dir, "test-*.parquet")))

            self._dataset = load_dataset(
                "parquet",
                data_files=train_files,
                split="train"
            )
        if split == 'valid':
            data_dir = r"D:\hf_datasets\hub\datasets--nguyendv02--ViMD_Dataset\snapshots\3a5b30157034e7eadd5c75fae1a820c6f9383398\data"
            train_files = sorted(glob.glob(os.path.join(data_dir, "valid-*.parquet")))

            self._dataset = load_dataset(
                "parquet",
                data_files=train_files,
                split="train"
            )
#-------------------------------------------------------------KHÚC NÀY ĐÃ SỬA--------------------------------------------------
        # Lọc các cột cần thiết để tối ưu bộ nhớ nếu cần
        self._dataset = self._dataset.select_columns(["audio", "text"])
        self._dataset = self._dataset.cast_column("audio", Audio(decode=False))

        self._valid_indices = list(range(len(self._dataset)))

        if self.filter_bad_samples:
            self._valid_indices = self._build_valid_indices()

    def _build_valid_indices(self):

        valid = []

        for idx in tqdm(range(len(self._dataset)), desc=f"Filtering {self.split}"):

            try:
                waveform, sr, transcript = self._load_raw(idx)

                if transcript is None:
                    continue

                if len(transcript.strip()) == 0:
                    continue

                duration = waveform.shape[-1] / sr

                if duration < self.min_duration_sec:
                    continue

                silence_ratio = (waveform.abs() < 1e-4).float().mean().item()

                if silence_ratio >= self.max_silence_ratio:
                    continue

                valid.append(idx)

            except Exception:
                continue

        print(f"Kept {len(valid)}/{len(self._dataset)} samples")

        return valid

    def _load_raw(self, idx: int) -> Tuple[torch.Tensor, int, str]:

        item = self._dataset[idx]

        transcript = item.get("text", "")

        audio_bytes = item["audio"]["bytes"]

        with io.BytesIO(audio_bytes) as f:
            array, sr = sf.read(f, dtype="float32")

        waveform = torch.from_numpy(array)

        if waveform.ndim > 1:
            waveform = waveform.mean(dim=-1)

        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)

        if self.target_sr and sr != self.target_sr:
            waveform = torchaudio.functional.resample(
                waveform,
                sr,
                self.target_sr
            )
            sr = self.target_sr

        return waveform, sr, transcript

    def __len__(self):

        return len(self._valid_indices)

    def __getitem__(self, n: int):

        idx = self._valid_indices[n]

        waveform, sr, transcript = self._load_raw(idx)

        transcript = normalize_transcript(transcript)

        return waveform, sr, transcript

In [ ]:
import torch
import time
import gc
import os
import tempfile
import soundfile as sf
from qwen_asr import Qwen3ASRModel
from datasets import load_dataset, Audio
from jiwer import wer

MODEL_ID = "Qwen/Qwen3-ASR-0.6B"
DATASET_ID = "nguyendv02/ViMD_Dataset"
SPLIT = "valid"
SAMPLING_RATE = 16000

def main():
    print(f"Đang tải model {MODEL_ID}...")
    model = Qwen3ASRModel.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        device_map="cuda:0",
        max_inference_batch_size=32,
        max_new_tokens=256,
    )

    print(f"Đang kết nối tới dataset {DATASET_ID}...")
    dataset = ViMD(split="valid")

    predictions = []
    references = []
    total_inference_time = 0.0
    count = 0

    print("Bắt đầu Inference (Streaming + Temp File Bridge)...")

    try:
        for item in tqdm(dataset, desc="Inference", unit="sample"):
            waveform, sr, ground_truth = item

            if not ground_truth:
                continue

            ground_truth = str(ground_truth).strip()
            count += 1

            # tensor -> numpy
            audio_array = waveform.squeeze().cpu().numpy()

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
                sf.write(tmp_file.name, audio_array, sr)
                temp_audio_path = tmp_file.name

            start_time = time.time()

            with torch.no_grad():
                results = model.transcribe(audio=temp_audio_path, language=None)
                output_text = results[0].text

            end_time = time.time()

            if os.path.exists(temp_audio_path):
                os.remove(temp_audio_path)

            predictions.append(normalize_transcript(output_text))
            references.append(ground_truth)

            total_inference_time += (end_time - start_time)

            if count % 50 == 0:
                print(f"Đã xong {count} mẫu. (Avg Time: {total_inference_time/count:.4f}s)")

    except Exception as e:
        print(f"\n Lỗi phát sinh: {e}")

    if count > 0:
        avg_wer = wer(references, predictions)
        avg_time = total_inference_time / count

        print("\n" + "="*50)
        print("KẾT QUẢ ĐÁNH GIÁ CUỐI CÙNG")
        print("="*50)
        print(f"- Tổng số mẫu đã xử lý : {count}")
        print(f"- WER trung bình        : {avg_wer:.4f} ({avg_wer*100:.2f}%)")
        print(f"- Time per sample       : {avg_time:.4f} giây")
        print("="*50)

    del model
    gc.collect()
    torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

Đang tải model Qwen/Qwen3-ASR-0.6B...


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Đang kết nối tới dataset nguyendv02/ViMD_Dataset...
Loading ViMD dataset split: valid...


Generating train split: 1900 examples [00:25, 75.47 examples/s]
Filtering valid: 100%|██████████| 1900/1900 [00:42<00:00, 44.84it/s]


Kept 1900/1900 samples
Bắt đầu Inference (Streaming + Temp File Bridge)...


KeyboardInterrupt: 